In [17]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langsmith import Client
from langchain.tools import tool
import requests

In [18]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [19]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [20]:
search_tool = TavilySearchResults(max_results=2)

In [21]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [22]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'The Latest AI News and Breakthroughs That Matter Most | News',
  'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': 'Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice and audio recognition capabilities with LivePerson\'s digital messaging products. Management has set 2027 revenue targets of $350 million to $400 million and identified a $500 million cross-selling opportunity across the combined customer base. The deal positions SoundHound to compete more directly with Microsoft, Google, and Amazon across enterprise contact centers, healthcare, and financial services.\n\nSource: Simply Wall St ↗\n\n### (AI News) Nvidia\'s Memory Costs Soar 485% as Latest AI Systems Now Cost $7.8 Million to Build\n\nDate: May 21, 2026 [...] Source: 425 Business\n\n### (AI Innovation) Microsoft Launches $2.5 Billion Frontier Company for Enterprise AI Deployment\n\nDate: July 2, 2026\n

In [23]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [24]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she kept giving him too many bytes!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 13, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EYvU5aKxoxnJRxn5b9w09Biowvw54', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13b18-95e1-7fd2-b533-397a0028cd5c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 20, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_d

In [25]:
# ==========================================
# PROMPT
# ==========================================

prompt = Client().pull_prompt(
    "hwchase17/react",
    include_model=False,
    dangerously_pull_public_prompt=True,
)
prompt = PromptTemplate.from_template(
    "Answer in the same language as the user's question. "
    "Keep proper nouns and technical terms unchanged when appropriate.\n\n"
    + prompt.template,
    template_format=prompt.template_format,
    metadata=prompt.metadata,
)

In [26]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template="Answer in the same language as the user's question. Keep proper nouns and technical terms unchanged when appropriate.\n\nAnswer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratch

In [27]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [28]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [29]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [30]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Quelle est la capitale de la France ? "
        "Quelle est sa population ?"
        "Quel temps y fait-il ?"
    )
})




> Entering new AgentExecutor chain...
Je dois utiliser la fonction get_weather_data pour obtenir des informations sur la météo de la capitale de la France.
Action: get_weather_data
Action Input: "Paris"City: Paris
Temperature: 21°C
Weather: Sunny
Humidity: 33%Je dois maintenant utiliser la fonction tavily_search_results_json pour obtenir des informations sur la population de la capitale de la France.
Action: tavily_search_results_json
Action Input: "Population of Paris, France"[{'title': 'Paris - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Paris', 'content': 'The population of the City of Paris was 2,102,650 in January 2023, down from 2,165,423 in January 2022. Between 2013 and 2023, the population fell by 122,919, or about five percent. The Mayor of Paris, Anne Hidalgo, declared that this illustrated the "de-densification" of the city, creating more green space and less crowding. Despite the drop, Paris remains the most densely populated city in Europe, with 252 residents per 

In [31]:
print(response["output"])

La capitale de la France est Paris. Sa population est d'environ 2,102,650 habitants. Il fait actuellement ensoleillé avec une température de 21°C.
